In [1]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

In [2]:
pretrained_model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(pretrained_model_name)
model = BertForSequenceClassification.from_pretrained(pretrained_model_name, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [4]:
train_texts = [
    "I love this product, it's amazing!",  # Positive
    "Absolutely fantastic experience, will buy again!",  # Positive
    "Worst purchase ever. Completely useless.",  # Negative
    "I hate this item, it doesn't work!",  # Negative
    "The quality is top-notch, highly recommend!",  # Positive
    "Terrible service, never coming back.",  # Negative
    "This is the best thing I've ever bought!",  # Positive
    "Very disappointing. Waste of money.",  # Negative
    "Superb! Exceeded all my expectations.",  # Positive
    "Not worth the price at all.",  # Negative
]
train_labels = torch.tensor([1, 1, 0, 0, 1, 0, 1, 0, 1, 0]).to(device)

In [5]:
encoded_train = tokenizer(train_texts,
                          padding=True,
                          truncation=True,
                          max_length=128,
                          return_tensors='pt')
train_input_ids = encoded_train['input_ids'].to(device)
train_attention_masks = encoded_train['attention_mask'].to(device)

In [6]:
train_dataset = TensorDataset(
    train_input_ids, train_attention_masks, train_labels)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [7]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [8]:
epochs = 5
model.train()
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        batch_input_ids, batch_attention_masks, batch_labels = batch

        optimizer.zero_grad()
        outputs = model(input_ids=batch_input_ids, 
                        attention_mask=batch_attention_masks, 
                        labels=batch_labels)

        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(F.softmax(logits, dim=1), dim=1)
        correct += (preds == batch_labels).sum().item()
        total += batch_labels.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1 - Loss: 0.6399, Accuracy: 60.00%
Epoch 2 - Loss: 0.6139, Accuracy: 80.00%
Epoch 3 - Loss: 0.5048, Accuracy: 90.00%
Epoch 4 - Loss: 0.3828, Accuracy: 100.00%
Epoch 5 - Loss: 0.3012, Accuracy: 100.00%


In [9]:
torch.save(model.state_dict(), "fine_tuned_bert.pth")

In [10]:
model.load_state_dict(torch.load("fine_tuned_bert.pth"))
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [11]:
test_texts = [
    "This is a great product, I love it!",  # Positive
    "Horrible experience, I want a refund!",  # Negative
    "Highly recommended! Five stars.",  # Positive
    "Not worth it. I regret buying this.",  # Negative
]
test_labels = torch.tensor([1, 0, 1, 0]).to(device)

encoded_test = tokenizer(test_texts, 
                         padding=True, 
                         truncation=True, 
                         max_length=128, 
                         return_tensors='pt')
                         
test_input_ids = encoded_test['input_ids'].to(device)
test_attention_masks = encoded_test['attention_mask'].to(device)

In [12]:
model.eval()
with torch.no_grad():
    outputs = model(input_ids=test_input_ids,
                    attention_mask=test_attention_masks)
    predicted_labels = torch.argmax(outputs.logits, dim=1)

test_accuracy = (predicted_labels == test_labels).sum(
).item() / len(test_labels) * 100
print(f"\nTest Accuracy: {test_accuracy:.2f}%")

for text, label in zip(test_texts, predicted_labels):
    print(f'Text: {text}\nPredicted Label: {label.item()}\n')


Test Accuracy: 75.00%
Text: This is a great product, I love it!
Predicted Label: 1

Text: Horrible experience, I want a refund!
Predicted Label: 1

Text: Highly recommended! Five stars.
Predicted Label: 1

Text: Not worth it. I regret buying this.
Predicted Label: 0

